In [27]:
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install langchain-groq
!pip install pypdf
!pip install langchain-text-splitter
!pip install langchain-classic
!pip install faiss-gpu
!pip install rank_bm25

ERROR: Could not find a version that satisfies the requirement langchain-text-splitter (from versions: none)
ERROR: No matching distribution found for langchain-text-splitter


In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Point to the official classic bridge package
from langchain_classic.chains import RetrievalQAWithSourcesChain

#from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("All modules imported successfully!")

All modules imported successfully!


In [13]:
# Set your exact Groq API key
os.environ['GROQ_API_KEY'] = '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'

In [15]:
#loading files
pdf_files = [
    "semantic.pdf",
    "semantic1.pdf",
    "semantic2.pdf",
    "semantic3.pdf",
    "semantic4.pdf"]

documents = []
print("Loading your PDF files...")

for pdf in pdf_files:
    if os.path.exists(pdf):
        loader = PyPDFLoader(pdf)
        documents.extend(loader.load())
        print(f" -> Successfully loaded: {pdf}")
    else:
        print(f" ⚠️ Warning: '{pdf}' was not found in this folder. Skipping it.")

print(f"Total PDF pages loaded into memory: {len(documents)}")

Loading your PDF files...
 -> Successfully loaded: semantic.pdf
 -> Successfully loaded: semantic1.pdf
 -> Successfully loaded: semantic2.pdf
 -> Successfully loaded: semantic3.pdf
 -> Successfully loaded: semantic4.pdf
Total PDF pages loaded into memory: 62


In [23]:
#split data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20)
chunks =text_splitter.split_documents(documents)
#print(chunks)


In [24]:
#embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
#faiss databasae
vector_store = FAISS.from_documents(chunks, embedding_model)

In [28]:
#bm25 search
bm25_retriever = BM25Retriever.from_documents(chunks)

In [29]:
#ensemble retriver
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_store.as_retriever()],
    weights=[0.5, 0.5])

In [31]:
#intializing groq
llm = ChatGroq(
        api_key=os.environ['GROQ_API_KEY'],
        model="llama-3.1-8b-instant"
    )

In [32]:
#prompts
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer only from the provided context.

If the answer is not found in the context, say:
"I could not find the answer in the uploaded documents."

Context:
{context}

Question:
{input}

Answer:
""")

In [33]:
chain = RetrievalQAWithSourcesChain.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=ensemble_retriever,
    )

In [34]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [37]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    ensemble_retriever,
    document_chain
)

In [38]:
response = retrieval_chain.invoke(
    {"input": "What is semantic search?"}
)

print(response["answer"])

I could not find the answer in the uploaded documents.


In [39]:
docs = ensemble_retriever.invoke("What is semantic search?")

print("Number of docs:", len(docs))

for i, doc in enumerate(docs):
    print(f"\nDoc {i+1}")
    print(doc.page_content[:500])

Number of docs: 7

Doc 1
et al., 2018).
STS-B The Semantic Textual Similarity Bench-
mark is a collection of sentence pairs drawn from
news headlines and other sources (Cer et al.,
2017). They were annotated with a score from 1
to 5 denoting how similar the two sentences are in
terms of semantic meaning.
MRPC Microsoft Research Paraphrase Corpus
consists of sentence pairs automatically extracted
from online news sources, with human annotations
for whether the sentences in the pair are semanti-
cally equivalent (Dolan an

Doc 2
References
[1] Y . Bengio, P. Simard, and P. Frasconi. Learning long-term dependen-
cies with gradient descent is difﬁcult. IEEE Transactions on Neural
Networks, 5(2):157–166, 1994.
[2] C. M. Bishop. Neural networks for pattern recognition . Oxford
university press, 1995.
[3] W. L. Briggs, S. F. McCormick, et al. A Multigrid Tutorial. Siam,
2000.
[4] K. Chatﬁeld, V . Lempitsky, A. Vedaldi, and A. Zisserman. The devil
is in the details: an evaluation of recent fea